# UruTracker - Radar de Emenda Pix Parada

Identifica municipios com emenda Pix aprovada e obra parada, ranqueados por urgencia.
Dados: API Transferegov | Periodo: 2024+

## Parte 1 - Carga e Integracao dos Dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import date

DADOS = Path("../../../data_extraction")
HOJE = pd.Timestamp(date.today())

In [ ]:
plano_acao     = pd.read_csv(DADOS / "plano_acao_especial.csv", encoding="utf-8-sig")
plano_trabalho = pd.read_csv(DADOS / "plano_trabalho_especial.csv", encoding="utf-8-sig")
executor       = pd.read_csv(DADOS / "executor_especial.csv", encoding="utf-8-sig")
finalidade     = pd.read_csv(DADOS / "finalidade_especial.csv", encoding="utf-8-sig")

print(plano_acao.shape, plano_trabalho.shape, executor.shape, finalidade.shape)

In [ ]:
plano_trabalho["data_fim_execucao_plano_trabalho"] = pd.to_datetime(
    plano_trabalho["data_fim_execucao_plano_trabalho"], errors="coerce"
)

plano_acao["valor_total"] = (
    plano_acao["valor_custeio_plano_acao"].fillna(0)
    + plano_acao["valor_investimento_plano_acao"].fillna(0)
)

finalidade_agg = (
    finalidade.groupby("id_executor")["area_politica_publica_pt"]
    .apply(lambda x: " | ".join(sorted(x.dropna().unique())))
    .reset_index()
    .rename(columns={"area_politica_publica_pt": "setor"})
)

executor_primeiro = executor.drop_duplicates("id_plano_acao", keep="first")

In [ ]:
df = plano_acao.merge(plano_trabalho, on="id_plano_acao", how="left")
df = df.merge(executor_primeiro[["id_plano_acao", "id_executor", "objeto_executor"]], on="id_plano_acao", how="left")
df = df.merge(finalidade_agg, on="id_executor", how="left")

print(f"Total: {len(df)} emendas")
df.head(3)

## Parte 2 - Classificacao e KPIs

### Classificacao de emendas paradas

Uma emenda e considerada **parada** se atende ao menos uma das condicoes:
- **A:** situacao_plano_trabalho == APROVADO
- **C:** prazo de execucao vencendo em menos de 90 dias

Urgencia: PRAZO_CRITICO > APROVADO_PENDENTE

In [ ]:
CONCLUIDOS = {"CONCLUIDO", "CONCLUIDO_NT_TCU", "CONCLUIDO_PRESTACAO_CONTAS"}

cond_A = df["situacao_plano_trabalho"] == "APROVADO"
cond_C = (
    (df["data_fim_execucao_plano_trabalho"] < (HOJE + pd.Timedelta(days=90)))
    & (~df["situacao_plano_trabalho"].isin(CONCLUIDOS))
)

df["parado"] = cond_A | cond_C
df_parado = df[df["parado"]].copy()

def urgencia(row):
    if pd.notna(row["data_fim_execucao_plano_trabalho"]) and \
       row["data_fim_execucao_plano_trabalho"] < (HOJE + pd.Timedelta(days=90)):
        return "PRAZO_CRITICO"
    return "APROVADO_PENDENTE"

df_parado["urgencia"] = df_parado.apply(urgencia, axis=1)
df_parado["dias_para_prazo"] = (df_parado["data_fim_execucao_plano_trabalho"] - HOJE).dt.days
df_parado["rank"] = df_parado["urgencia"].map({"PRAZO_CRITICO": 1, "APROVADO_PENDENTE": 2})
df_parado = df_parado.sort_values(["rank", "dias_para_prazo"])

print(f"Emendas paradas: {len(df_parado)} ({len(df_parado)/len(df)*100:.1f}%)")
df_parado["urgencia"].value_counts()

In [ ]:
print(f"Total emendas paradas : {len(df_parado)}")
print(f"Valor total           : R$ {df_parado['valor_total'].sum()/1e6:.1f}M")
print(f"Municipios            : {df_parado['nome_beneficiario_plano_acao'].nunique()}")
print(f"Prazo critico (<90d)  : {(df_parado['urgencia']=='PRAZO_CRITICO').sum()}")

## Parte 3 - Analises e Visualizacoes

In [ ]:
por_uf = df_parado.groupby("uf_beneficiario_plano_acao")["id_plano_acao"].count().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
por_uf.plot(kind="bar", ax=ax)
ax.set_title("Emendas paradas por UF")
ax.set_xlabel("UF")
ax.set_ylabel("Quantidade")
plt.tight_layout()
plt.show()

In [ ]:
por_setor = df_parado.dropna(subset=["setor"]).groupby("setor")["id_plano_acao"].count().sort_values().tail(15)

fig, ax = plt.subplots(figsize=(10, 6))
por_setor.plot(kind="barh", ax=ax)
ax.set_title("Top 15 setores com emendas paradas")
plt.tight_layout()
plt.show()

In [ ]:
por_urgencia = df_parado["urgencia"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
por_urgencia.plot(kind="bar", ax=axes[0])
axes[0].set_title("Quantidade por urgencia")

por_urgencia.plot(kind="pie", ax=axes[1], autopct="%1.1f%%")
axes[1].set_title("Proporcao por urgencia")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
colunas = ["urgencia", "nome_beneficiario_plano_acao", "uf_beneficiario_plano_acao",
           "setor", "objeto_executor", "valor_total", "nome_parlamentar_emenda_plano_acao", "dias_para_prazo"]

df_leads = df_parado[colunas].dropna(subset=["dias_para_prazo"]).copy()
df_leads["dias_para_prazo"] = df_leads["dias_para_prazo"].astype(int)

print(f"Leads com prazo definido: {len(df_leads)}")
df_leads.head(20)